In [ ]:
#manual hyperparameter tuning is here! production ready!
!pip install quick-sentiments==0.5.4 


In [ ]:
!pip show quick_sentiments #check for latest version

In [ ]:
import polars as pl

# here I have three python script I built to pre_process the data and running the pipeline
# you can find the code in the tools/preprocess.py file
# you can find  the code in the tools/pipeline.py file
# the pre_process function is used to clean the text data, there are various options available, please check the tools/preprocess.py file for details
# the run_pipeline function is used to run the sentimental analysis pipeline, it takes the training data and the vectorizer and machine learning methods as input, and returns the results
from quick_sentiments import pre_process_nltk
from quick_sentiments import pre_process_spacy
from quick_sentiments import run_pipeline
from quick_sentiments import make_predictions
from quick_sentiments import evaluate_performance

### Training Dataset


In [ ]:
# this template uses csv files 
# column names can be set in Python but this template does not automatically update the column for the demo 
# however, the function will give you the option to tell column names for the text and label data

df_train = pl.read_csv("demo/training_data/train.csv",encoding='ISO-8859-1') 
print(f"Dataset shape: {df_train.shape[0]} rows and {df_train.shape[1]} columns")


### DEMO

In [ ]:
df_train.head()
# randomly select only 10% of the data since the dataset is large
#RUN ONLY ONCE
df_train = df_train.sample(fraction=0.1, shuffle=True, seed=42) 
df_train.head(5)

The dataset is for training. The sentiments are already labeled. This will allow us to train a model that can predict sentiments on new data.


In [ ]:
# you can use the pre_process function to clean the text data
response_column = "reviewText" # this is the column name for the text data, feel free to change it to your text column name
sentiment_column = "sentiment" # this is the column name for the sentiment data, feel free to change it to your sentiment column name


In [ ]:
# make changes as necessary
# inside the map_elements, add  the parameters [pre_process(x, parameters_to_be_added)] and set it True/False if it differs from the defualt value
# check the tools/preprocess.py file for the parameters and their default values
# some of the parameters are remove_brackets, remove_stopwords, remove_punctuation, remove_numbers, remove_emojis, remove_urls, remove_html_tags, lemmatize, stem, lowercase
df_train = pre_process_nltk(df_train, text_column=response_column, new_column_name="cleaned_text_nltk")


In [ ]:
df_train = pre_process_spacy(df_train, text_column=response_column, new_column_name="cleaned_text_spacy")

In [ ]:
df_train.head()

In [ ]:
#### 6 plus text representation / vectorizer methods available 
#### in the function run_pipeline (in python cell below), we shall make use of this, write the words inside [ ] for the methods you want to use
#### 1. Bag of Words [BOW] 
#### 2. Term Frequency [tf]
#### 3. TF -IDF    [tfidf]
#### 4. Word Embedding using Word2Vec (you can use other packages with slight changes) [wv] 
         # Word Embedding uses defualt 300 values; this will take some time to run
#### 5. Glove (you can use other packages with slight changes) [glove_25,glove_50, glove_100, gl0ve_200]
#### 6. Hugging Face Transformers (you can use other packages with slight changes) [transformer]. You have to download hugging face transformer models to use this method, check the tools/pipeline.py file for details

In [ ]:
#### 6 there are also five machine learning methods that can be used
#### 1. Logistic Regression [logit]
#### 2. Random forest (recommended) (rf)
#### 3. XGBoosting  [XGB](word embedding and XGBoost may take long time to complete, combination of both is not recommended in local machine)
#### 4. Naive Bayes [nb]
#### 5. Neural Network [nn] (this will take some time to run, and may run out of memory if the dataset is large, so be careful when using this method)
#### 6. Tensorflow/Keras [tf, tensorflow, keras] (this will take some time to run, and may run out of memory if the dataset is large, so be careful when using this method)


In [ ]:
# example parameters for all
#Logit
custom_logit = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['liblinear', 'lbfgs', 'saga'],
    'class_weight': [None, 'balanced'],
    'max_iter': [500, 1000, 2000]
}
#Random Forest
custom_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced', 'balanced_subsample']
}
#XGBoost
custom_xgb = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2] # Minimum loss reduction required to make a further partition
}
#Neural Network
custom_nn = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'sgd'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01] 
}
#Tensorflow/Keras
custom_tf = {
    'model__hidden_units': [64, 128, 256],
    'model__dropout_rate': [0.2, 0.3, 0.5],
    'batch_size': [32, 64, 128],
    'epochs': [10, 20] 
}
#Naive Bayers Multinomial
custom_nb_multinomial = {
    'alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0], # Smoothing parameter
    'fit_prior': [True, False]
}
#Naive Bayes Gaussian (If using Word2Vec, GloVe, or Hugging Face (Gaussian)
custom_nb_gaussian = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5] # Portion of the largest variance of all features
}


In [ ]:
# this is the example of how to use the function
# you can change the vectorizer_name and model_name to the ones you want to use
# for now we will use word embedding and tensorflow
# write the name of your columns in the text_column_name and sentiment_column_name
# the text_column_name is the column name of the text data, and sentiment_column_name is

# run_pipeline function will return the dataframe with the vectorized text, vectorizer used  and the model
# it will also print the results of the model, including the accuracy and F1 score
# note, even without hyperparameter tuning, the model is getting over 70% accuracy in my test
# there may not be a need to perform hyperparameter tuning, but you can set perform_tuning to True if you want to do that
model = run_pipeline(
    vectorizer_name="BOW", # BOW, tf, tfidf, wv,  glove_25,glove_50, glove_100, gl0ve_200,
    model_name="rf", # logit, rf, XGB, nb, nn, tf for tensorflow models .#XGB takes long time, can not recommend using it on normal case
    df=df_train,
    text_column_name="cleaned_text_spacy",  # this is the column name of the text data, 
    sentiment_column_name = "sentiment",
    perform_tuning = True, # make this true if you want to perform hyperparameter tuning, it will take longer time and 
    param_grid=custom_rf,
    interactive=True, 
    random_state=259
)

In [ ]:
evaluate_performance(model["y_test"], model["y_prob"],positive_label=1)

In [ ]:
## the model is a dictionary that contains the results of the model, including the accuracy and F1 score

# you can access the results using the keys of the dictionary
print("Vectorizer used: ", model["vectorizer_name"])
print("Model used: ", model["model_object"])
print("Accuracy: ", model["accuracy"])



### New Dataset for prediction
You can use the same format as the training dataset, but ensure that it contains the "Response" column for text data. The "Sentiment" column is optional for prediction datasets, as it will be generated by the model.
Make sure the dataset is saved in the "New Data" folder and is in CSV format.

In [ ]:
new_data = pl.read_csv("demo/new_data/test.csv",encoding='ISO-8859-1') #keep your file here
print(new_data.shape)
new_data= new_data.sample(fraction=0.25, shuffle=True, seed=42)
print(new_data.shape)

In [ ]:
new_data = pre_process_nltk(new_data, text_column=response_column, new_column_name="cleaned_text")
new_data.head()

In [ ]:
make_predictions(
    new_data=new_data,
    text_column_name="cleaned_text",  # this is the column name of the text data,
    prediction_column_name="sentiment_predictions",  # Optional custom name
    trained_results=model
)